Extraction et traitement des données

Soit, télécharger les données, puis garder seulement les variables voulues, puis ajouter celles qu'on souhaite.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)



In [2]:
DATA_DIR = Path("../data")

per100 = pd.read_csv(DATA_DIR / "Per_100_Poss.csv")
player_info = pd.read_csv(DATA_DIR / "Player_Season_Info.csv")
team_summary = pd.read_csv(DATA_DIR / "Team_Summaries_franchise.csv")
team_abbrev = pd.read_csv(DATA_DIR / "Team_Abbrev_franchise.csv")
all_star = pd.read_csv(DATA_DIR / "All-Star_Selections.csv")
end_season = pd.read_csv(DATA_DIR / "End_of_Season_Teams.csv")
team_p100 = pd.read_csv(DATA_DIR / "Team_Stats_Per_100_Poss_franchise.csv")
op_team_p100 =  pd.read_csv(DATA_DIR / "Opponent_Stats_Per_100_Poss_franchise.csv")

In [3]:
datasets = {
    "Per100": per100,
    "Player_Info": player_info,
    "Team_Summary": team_summary,
    "Team_Abbrev": team_abbrev,
    "All_Star": all_star,
    "End_Season": end_season,
    'team_p100' : team_p100,
    'op_team_p100' : op_team_p100,
}

for name, df in datasets.items():
    print(name)
    print("Shape :", df.shape)
    print("Colonnes :", df.columns.tolist())

Per100
Shape : (27692, 34)
Colonnes : ['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'g', 'gs', 'mp', 'fg_per_100_poss', 'fga_per_100_poss', 'fg_percent', 'x3p_per_100_poss', 'x3pa_per_100_poss', 'x3p_percent', 'x2p_per_100_poss', 'x2pa_per_100_poss', 'x2p_percent', 'e_fg_percent', 'ft_per_100_poss', 'fta_per_100_poss', 'ft_percent', 'orb_per_100_poss', 'drb_per_100_poss', 'trb_per_100_poss', 'ast_per_100_poss', 'stl_per_100_poss', 'blk_per_100_poss', 'tov_per_100_poss', 'pf_per_100_poss', 'pts_per_100_poss', 'o_rtg', 'd_rtg']
Player_Info
Shape : (33339, 8)
Colonnes : ['season', 'lg', 'player', 'player_id', 'age', 'team', 'pos', 'experience']
Team_Summary
Shape : (1907, 11)
Colonnes : ['season', 'lg', 'team', 'abbreviation', 'playoffs', 'age', 'w', 'l', 'pw', 'pl', 'franchise_id']
Team_Abbrev
Shape : (1818, 6)
Colonnes : ['season', 'lg', 'team', 'abbreviation', 'playoffs', 'franchise_id']
All_Star
Shape : (2058, 6)
Colonnes : ['player', 'player_id', 'team', 'season', 'lg

In [4]:
print("Per100 :", per100["season"].min(), "→", per100["season"].max())
print("Player info :", player_info["season"].min(), "→", player_info["season"].max())
print("Team summary :", team_summary["season"].min(), "→", team_summary["season"].max())

Per100 : 1974 → 2026
Player info : 1947 → 2026
Team summary : 1947 → 2026


Debut des per 100 en 1974, et selection des siasons avec seulement 82 matchs pour eviter des billets

In [5]:
numeric_cols = per100.select_dtypes(
    include=["number"]
).columns.tolist()

numeric_cols

per100[numeric_cols] = per100[numeric_cols].apply(
    pd.to_numeric,
    errors="coerce"
)

In [6]:
per100 = per100[per100["lg"] == "NBA"].copy()

Nettoyage des données

In [7]:
def clean_basic(df):
    df = df.copy()

    # espaces inutiles dans les noms
    df.columns = df.columns.str.strip()

    # suppression des lignes complètement vides
    df = df.dropna(how="all")

    # suppression des espaces autour des chaînes
    for col in df.select_dtypes(include=["object"]).columns:
        df[col] = df[col].str.strip()

    # suppression des lignes 2TM, 3TM, etc.
    if "team" in df.columns:
        df = df[~df["team"].str.match(r"^\d+TM$", na=False)]

    # garder uniquement les saisons à partir de 1974 inclus
    if "season" in df.columns:
        df = df[df["season"] >= 1974]

    return df


In [8]:
per100 = clean_basic(per100)
player_info = clean_basic(player_info)
team_summary = clean_basic(team_summary)
team_abbrev = clean_basic(team_abbrev)
all_star = clean_basic(all_star)
end_season = clean_basic(end_season)
team_p100 = clean_basic(team_p100)
op_team_p100 =  clean_basic(op_team_p100)

C:\Users\marsr\AppData\Local\Temp\ipykernel_36840\97752745.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:
C:\Users\marsr\AppData\Local\Temp\ipykernel_36840\97752745.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/

In [9]:
# Saisons où les équipes ont joué 82 matchs
team_summary["g"] = team_summary["l"] + team_summary["w"]
team_summary["win_pct"] = team_summary["w"] / team_summary["g"]
target = team_summary[
    ["franchise_id", "season", "w", "g",'win_pct']
].copy()

target["win_pct"] = target["w"] / target["g"]

In [10]:
datasets = {
    "team_p100": team_p100,
    "op_team_p100": op_team_p100
}

for name, df in datasets.items():

    # Années >= 1974
    df = df[df["season"] >= 1974].copy()

    # Supprimer les lignes agrégées des joueurs ayant joué
    # pour plusieurs équipes dans la même saison
    df = df[
        ~df["team"].astype(str).str.contains(r"TM$", na=False)
    ].copy()

    # Vérifier la clé
    key = [ "team", "season"]

    duplicates = (
        df
        .groupby(key)
        .size()
        .reset_index(name="n")
        .query("n > 1")
    )

    print(f"\n--- {name} ---")
    print(f"Nombre de lignes : {len(df):,}")
    print(f"Doublons sur {key} : {len(duplicates):,}")

    if len(duplicates) > 0:
        display(duplicates.head(20))
    else:
        print("La clé player_id + team + season est unique.")

    # Mettre à jour le DataFrame
    datasets[name] = df

team_p100 = datasets["team_p100"]
op_team_p100 = datasets["op_team_p100"]


--- team_p100 ---
Nombre de lignes : 1,462
Doublons sur ['team', 'season'] : 0
La clé player_id + team + season est unique.

--- op_team_p100 ---
Nombre de lignes : 1,462
Doublons sur ['team', 'season'] : 0
La clé player_id + team + season est unique.


Création des scores attribués (all-star et all-nba (defensive-team))

In [11]:
all_star_score = (
    all_star[["player_id", "season"]]
    .drop_duplicates()
    .assign(all_star_score=1)
)

all_star_score.head()

,player_id,season,all_star_score
0,barnesc01,2026,1
1,bookede01,2026,1
2,cunnica01,2026,1
3,durenja01,2026,1
4,edwaran01,2026,1


In [12]:
# Garder uniquement les sélections All-NBA
all_nba = end_season[
    end_season["type"] == "All-NBA"
].copy()

# Attribuer un score selon l'équipe All-NBA
all_nba["all_nba_score"] = all_nba["number_tm"].map({
    "1st": 3,
    "2nd": 2,
    "3rd": 1
})

# Garder uniquement ce dont on a besoin
all_nba_score = all_nba[
    ["player_id", "season", "all_nba_score"]
].copy()

# Vérification
display(all_nba_score.head(5))

,player_id,season,all_nba_score
10,jokicni01,2025,3
11,antetgi01,2025,3
12,tatumja01,2025,3
13,gilgesh01,2025,3
14,mitchdo01,2025,3


In [13]:
player_awards = pd.merge(
    all_star_score,
    all_nba_score,
    on=["player_id", "season"],
    how="outer"
).fillna(0)

display(player_awards.head(5))

,player_id,season,all_star_score,all_nba_score
0,abdulka01,1974,1.0,3.0
1,abdulka01,1975,1.0,0.0
2,abdulka01,1976,1.0,3.0
3,abdulka01,1977,1.0,3.0
4,abdulka01,1978,0.0,2.0


Il faut maintenant merge celui avec les poids qu'on attribue, personellement un lag de 1 an vaut un poid de 1, d il y a 2 ans vaut 0.75 et 3 ans 0.5

In [14]:
player_awards_team = player_awards.merge(
    player_info[["player_id", "season", "team"]],
    on=["player_id", "season"],
    how="left"
)

In [15]:
display(player_awards_team.head(5))

,player_id,season,all_star_score,all_nba_score,team
0,abdulka01,1974,1.0,3.0,MIL
1,abdulka01,1975,1.0,0.0,MIL
2,abdulka01,1976,1.0,3.0,LAL
3,abdulka01,1977,1.0,3.0,LAL
4,abdulka01,1978,0.0,2.0,LAL


In [16]:
team_awards = (
    player_awards_team
    .groupby(["team", "season"], as_index=False)
    .agg(
        all_star_score=("all_star_score", "sum"),
        all_nba_score=("all_nba_score", "sum")
    )
)

display(
    team_awards[
        team_awards["team"] == "MIL"
    ].head(10)
)

,team,season,all_star_score,all_nba_score
484,MIL,1974,1.0,3.0
485,MIL,1975,3.0,0.0
486,MIL,1976,2.0,0.0
487,MIL,1978,1.0,0.0
488,MIL,1979,1.0,3.0
489,MIL,1980,1.0,2.0
490,MIL,1981,1.0,2.0
491,MIL,1982,2.0,2.0
492,MIL,1983,2.0,3.0
493,MIL,1984,1.0,2.0


Creation de fonctions pour faire des sommes et ecart types ponderées

In [17]:
import numpy as np

def weighted_mean_std(row, cols, weights):

    values = row[cols].values.astype(float)

    mask = ~np.isnan(values)

    if mask.sum() == 0:
        return pd.Series([np.nan, np.nan])

    values = values[mask]
    weights = np.array(weights)[mask]

    weighted_mean = np.average(
        values,
        weights=weights
    )

    weighted_var = np.average(
        (values - weighted_mean) ** 2,
        weights=weights
    )

    weighted_std = np.sqrt(weighted_var)

    return pd.Series([
        weighted_mean,
        weighted_std
    ])

weights = [1.0, 0.7, 0.4]

Création des variables precdente dans le temps

In [18]:
# Toutes les combinaisons team-season existantes
team_seasons = (
    team_p100[
        ["franchise_id", "season"]
    ]
    .drop_duplicates()
)

team_awards.rename(columns={'team': 'franchise_id'}, inplace=True)
# Ajouter les saisons sans awards
team_awards = team_seasons.merge(
    team_awards,
    on=["franchise_id", "season"],
    how="left"
)

# Une saison sans award = score de 0
team_awards[
    ["all_star_score", "all_nba_score"]
] = team_awards[
    ["all_star_score", "all_nba_score"]
].fillna(0)

# Trier
team_awards = team_awards.sort_values(
    ["franchise_id", "season"]
)

# Trier
team_awards = team_awards.sort_values(
    ["franchise_id", "season"]
)

# LAGS ALL-STAR
team_awards["all_star_prev1"] = (
    team_awards.groupby("franchise_id")["all_star_score"].shift(1)
)
team_awards["all_star_prev2"] = (
    team_awards.groupby("franchise_id")["all_star_score"].shift(2)
)
team_awards["all_star_prev3"] = (
    team_awards.groupby("franchise_id")["all_star_score"].shift(3)
)

# LAGS ALL-NBA
team_awards["all_nba_prev1"] = (
    team_awards.groupby("franchise_id")["all_nba_score"].shift(1)
)
team_awards["all_nba_prev2"] = (
    team_awards.groupby("franchise_id")["all_nba_score"].shift(2)
)
team_awards["all_nba_prev3"] = (
    team_awards.groupby("franchise_id")["all_nba_score"].shift(3)
)

In [19]:
# ALL-STAR
all_star_cols = [
    "all_star_prev1",
    "all_star_prev2",
    "all_star_prev3"
]

team_awards[["all_star_mean", "all_star_std"]] = (
    team_awards[all_star_cols].apply(
        lambda row: weighted_mean_std(
            row,
            all_star_cols,
            weights
        ),
        axis=1
    )
)


# ALL-NBA
all_nba_cols = [
    "all_nba_prev1",
    "all_nba_prev2",
    "all_nba_prev3"
]

team_awards[["all_nba_mean", "all_nba_std"]] = (
    team_awards[all_nba_cols].apply(
        lambda row: weighted_mean_std(
            row,
            all_nba_cols,
            weights
        ),
        axis=1
    )
)

In [20]:
display(
    team_awards[
        (team_awards["season"] == 2022)
    ]
)

,franchise_id,season,all_star_score,all_nba_score,all_star_prev1,all_star_prev2,all_star_prev3,all_nba_prev1,all_nba_prev2,all_nba_prev3,all_star_mean,all_star_std,all_nba_mean,all_nba_std
120,ATL,2022,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.333333,0.471405,0.000000,0.000000
121,BOS,2022,1.0,3.0,2.0,2.0,1.0,0.0,1.0,2.0,1.809524,0.392677,0.714286,0.764875
122,BRK,2022,2.0,2.0,3.0,0.0,1.0,1.0,0.0,0.0,1.619048,1.361938,0.476190,0.499433
124,CHA,2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
123,CHI,2022,2.0,2.0,2.0,0.0,0.0,0.0,0.0,0.0,0.952381,0.998866,0.000000,0.000000
125,CLE,2022,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000
126,DAL,2022,1.0,3.0,1.0,1.0,1.0,3.0,3.0,0.0,1.000000,0.000000,2.428571,1.178030
127,DEN,2022,1.0,3.0,1.0,1.0,1.0,3.0,2.0,3.0,1.000000,0.000000,2.666667,0.471405
128,DET,2022,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.190476,0.392677,0.190476,0.392677
129,GSW,2022,3.0,2.0,1.0,0.0,3.0,3.0,0.0,5.0,1.047619,1.045452,2.380952,1.838120


Il faut realiser la meme chose pour les statistiques team_per_100 et op_team_per_100

In [21]:
# Trier par équipe et saison
team_p100 = team_p100.sort_values(["team", "season"]).copy()
op_team_p100 = op_team_p100.sort_values(["team", "season"]).copy()


# Statistiques Per 100 à utiliser
p100_cols = p100_cols = [
    "fg_per_100_poss",
    "fga_per_100_poss",
    "fg_percent",
    "x3p_per_100_poss",
    "x3pa_per_100_poss",
    "x3p_percent",
    "x2p_per_100_poss",
    "x2pa_per_100_poss",
    "x2p_percent",
    "ft_per_100_poss",
    "fta_per_100_poss",
    "ft_percent",
    "orb_per_100_poss",
    "drb_per_100_poss",
    "trb_per_100_poss",
    "ast_per_100_poss",
    "stl_per_100_poss",
    "blk_per_100_poss",
    "tov_per_100_poss",
    "pf_per_100_poss",
    "pts_per_100_poss"
]

op_team_p100_cols = [
 'opp_fg_per_100_poss',
 'opp_fga_per_100_poss',
 'opp_fg_percent',
 'opp_x3p_per_100_poss',
 'opp_x3pa_per_100_poss',
 'opp_x3p_percent',
 'opp_x2p_per_100_poss',
 'opp_x2pa_per_100_poss',
 'opp_x2p_percent',
 'opp_ft_per_100_poss',
 'opp_fta_per_100_poss',
 'opp_ft_percent',
 'opp_orb_per_100_poss',
 'opp_drb_per_100_poss',
 'opp_trb_per_100_poss',
 'opp_ast_per_100_poss',
 'opp_stl_per_100_poss',
 'opp_blk_per_100_poss',
 'opp_tov_per_100_poss',
 'opp_pf_per_100_poss',
 'opp_pts_per_100_poss']

# TEAM PER 100
for col in p100_cols:

    team_p100[f"{col}_prev1"] = (
        team_p100.groupby("franchise_id")[col].shift(1)
    )

    team_p100[f"{col}_prev2"] = (
        team_p100.groupby("franchise_id")[col].shift(2)
    )

    team_p100[f"{col}_prev3"] = (
        team_p100.groupby("franchise_id")[col].shift(3)
    )

    prev_cols = [
        f"{col}_prev1",
        f"{col}_prev2",
        f"{col}_prev3"
    ]

    team_p100[[f"{col}_mean", f"{col}_std"]] = (
        team_p100[prev_cols].apply(
            lambda row: weighted_mean_std(
                row,
                prev_cols,
                weights
            ),
            axis=1
        )
    )


# OPPONENT PER 100
for col in op_team_p100_cols:

    op_team_p100[f"{col}_prev1"] = (
        op_team_p100.groupby("franchise_id")[col].shift(1)
    )

    op_team_p100[f"{col}_prev2"] = (
        op_team_p100.groupby("franchise_id")[col].shift(2)
    )

    op_team_p100[f"{col}_prev3"] = (
        op_team_p100.groupby("franchise_id")[col].shift(3)
    )

    prev_cols = [
        f"{col}_prev1",
        f"{col}_prev2",
        f"{col}_prev3"
    ]

    op_team_p100[[f"{col}_mean", f"{col}_std"]] = (
        op_team_p100[prev_cols].apply(
            lambda row: weighted_mean_std(
                row,
                prev_cols,
                weights
            ),
            axis=1
        )
    )

C:\Users\marsr\AppData\Local\Temp\ipykernel_36840\79374101.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  op_team_p100[[f"{col}_mean", f"{col}_std"]] = (
C:\Users\marsr\AppData\Local\Temp\ipykernel_36840\79374101.py:108: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  op_team_p100[[f"{col}_mean", f"{col}_std"]] = (


In [22]:
display(
    team_p100[
        (team_p100["franchise_id"] == "MIL") &
        (team_p100["season"] == 2018)
    ][[
        "team",
        "season",
        "pts_per_100_poss",
        "pts_per_100_poss_prev1",
        "pts_per_100_poss_prev2",
        "pts_per_100_poss_prev3",
        "pts_per_100_poss_mean",
        "pts_per_100_poss_std"
    ]]
)

,team,season,pts_per_100_poss,pts_per_100_poss_prev1,pts_per_100_poss_prev2,pts_per_100_poss_prev3,pts_per_100_poss_mean,pts_per_100_poss_std
256,Milwaukee Bucks,2018,109.8,109.1,104.3,102.7,106.280952,2.744973


In [23]:
display(
    op_team_p100[
        (op_team_p100["franchise_id"] == "MIL") &
        (op_team_p100["season"] == 2022)
    ][[
        "team",
        "season",
        "opp_pts_per_100_poss",
        "opp_pts_per_100_poss_prev1",
        "opp_pts_per_100_poss_prev2",
        "opp_pts_per_100_poss_prev3",
        "opp_pts_per_100_poss_mean",
        "opp_pts_per_100_poss_std"
    ]]
)

,team,season,opp_pts_per_100_poss,opp_pts_per_100_poss_prev1,opp_pts_per_100_poss_prev2,opp_pts_per_100_poss_prev3,opp_pts_per_100_poss_mean,opp_pts_per_100_poss_std
136,Milwaukee Bucks,2022,111.8,111.4,102.9,105.2,107.385714,3.910339


In [37]:
team_p100 = (
    team_p100
    .sort_values(["franchise_id", "season"])
    .reset_index(drop=True)
)


team_p100["playoffs"] = team_p100["playoffs"].astype(float)

team_p100["playoffs_prev1"] = (
    team_p100.groupby("franchise_id")["playoffs"].shift(1)
)

team_p100["playoffs_prev2"] = (
    team_p100.groupby("franchise_id")["playoffs"].shift(2)
)

team_p100["playoffs_prev3"] = (
    team_p100.groupby("franchise_id")["playoffs"].shift(3)
)

playoff_cols = [
    "playoffs_prev1",
    "playoffs_prev2",
    "playoffs_prev3"
]

team_p100[["playoffs_mean", "playoffs_std"]] = (
    team_p100[playoff_cols].apply(
        lambda row: weighted_mean_std(
            row,
            playoff_cols,
            weights
        ),
        axis=1
    )
)

In [32]:
display(
    team_p100.loc[
        (team_p100["franchise_id"] == "GSW") &
        (team_p100["season"] >= 2020),
        [
            "season",
            "playoffs",
            "playoffs_prev1",
            "playoffs_prev2",
            "playoffs_prev3",
            "playoffs_mean",
            "playoffs_std"
        ]
    ]
)

,season,playoffs,playoffs_prev1,playoffs_prev2,playoffs_prev3,playoffs_mean,playoffs_std
502,2020,0.0,1.0,1.0,1.0,1.000000,0.000000
503,2021,0.0,0.0,1.0,1.0,0.523810,0.499433
504,2022,1.0,0.0,0.0,1.0,0.190476,0.392677
505,2023,1.0,1.0,0.0,0.0,0.476190,0.499433
506,2024,0.0,1.0,1.0,0.0,0.809524,0.392677
507,2025,1.0,0.0,1.0,1.0,0.523810,0.499433
508,2026,0.0,1.0,0.0,1.0,0.666667,0.471405


On peut desormais creer le dataset final qu on utilisera pour entrainer les modeles avec les donneees voulues

In [33]:
team_awards = team_awards.rename(columns={"team": "franchise_id"})

# 1. BASE

team_p100_features = team_p100[
    ["franchise_id", "season"] +
    [
        col for col in team_p100.columns
        if col.endswith("_mean") or col.endswith("_std")
    ]
].copy()

nba_ml_dataset = team_p100_features.copy()


# 2. OPPONENT PER 100

op_team_p100_features = op_team_p100[
    ["franchise_id", "season"] +
    [
        col for col in op_team_p100.columns
        if col.endswith("_mean") or col.endswith("_std")
    ]
].copy()

nba_ml_dataset = nba_ml_dataset.merge(
    op_team_p100_features,
    on=["franchise_id", "season"],
    how="left"
)

# 3. AWARDS


team_awards_features = team_awards[
    [
        "franchise_id",
        "season",
        "all_star_mean",
        "all_star_std",
        "all_nba_mean",
        "all_nba_std"
    ]
].copy()

nba_ml_dataset = nba_ml_dataset.merge(
    team_awards_features,
    on=["franchise_id", "season"],
    how="left"
)


# 4. TARGET


target = team_summary[
    ["franchise_id", "season", "win_pct"]
].copy()

nba_ml_dataset = nba_ml_dataset.merge(
    target,
    on=["franchise_id", "season"],
    how="left"
)

In [34]:
nba_ml_dataset.columns.tolist()

['franchise_id',
 'season',
 'fg_per_100_poss_mean',
 'fg_per_100_poss_std',
 'fga_per_100_poss_mean',
 'fga_per_100_poss_std',
 'fg_percent_mean',
 'fg_percent_std',
 'x3p_per_100_poss_mean',
 'x3p_per_100_poss_std',
 'x3pa_per_100_poss_mean',
 'x3pa_per_100_poss_std',
 'x3p_percent_mean',
 'x3p_percent_std',
 'x2p_per_100_poss_mean',
 'x2p_per_100_poss_std',
 'x2pa_per_100_poss_mean',
 'x2pa_per_100_poss_std',
 'x2p_percent_mean',
 'x2p_percent_std',
 'ft_per_100_poss_mean',
 'ft_per_100_poss_std',
 'fta_per_100_poss_mean',
 'fta_per_100_poss_std',
 'ft_percent_mean',
 'ft_percent_std',
 'orb_per_100_poss_mean',
 'orb_per_100_poss_std',
 'drb_per_100_poss_mean',
 'drb_per_100_poss_std',
 'trb_per_100_poss_mean',
 'trb_per_100_poss_std',
 'ast_per_100_poss_mean',
 'ast_per_100_poss_std',
 'stl_per_100_poss_mean',
 'stl_per_100_poss_std',
 'blk_per_100_poss_mean',
 'blk_per_100_poss_std',
 'tov_per_100_poss_mean',
 'tov_per_100_poss_std',
 'pf_per_100_poss_mean',
 'pf_per_100_poss_std'

In [35]:
display(
    nba_ml_dataset[
        [
            "franchise_id",
            "season",
            "fg_per_100_poss_mean",
            "fg_per_100_poss_std",
            "all_star_mean",
            "all_nba_mean",
            "win_pct"
        ]
    ]
    .sample(20, random_state=42)
    .sort_values(["franchise_id", "season"])
)

,franchise_id,season,fg_per_100_poss_mean,fg_per_100_poss_std,all_star_mean,all_nba_mean,win_pct
67,BOS,1988,44.628571,0.177664,3.000000,4.428571,0.695122
168,CHA,1995,42.409524,0.427512,0.000000,0.000000,0.609756
218,CHI,1994,46.014286,0.532993,1.809524,4.142857,0.670732
413,DET,1984,40.823810,0.981692,1.619048,0.952381,0.597561
522,HOU,1987,44.047619,0.386214,1.809524,1.619048,0.512195
567,IND,1979,39.488235,0.344507,0.823529,0.000000,0.463415
576,IND,1988,41.523810,0.669687,0.000000,0.000000,0.463415
614,IND,2026,43.871429,1.724651,1.000000,0.809524,0.231707
649,LAC,2004,38.657143,1.018669,0.333333,0.000000,0.341463
887,NOH,2012,40.652381,0.600491,1.190476,0.857143,0.318182


In [41]:
display(
    team_p100[
        team_p100["franchise_id"] == "BRK"
    ][
        [
            "franchise_id",
            "abbreviation",
            "season",
            "playoffs",
            "playoffs_prev1",
            "playoffs_prev2",
            "playoffs_prev3",
            "playoffs_mean"
        ]
    ]
    .sort_values("season")
)

,franchise_id,abbreviation,season,playoffs,playoffs_prev1,playoffs_prev2,playoffs_prev3,playoffs_mean
106,BRK,NJN,1978,0.0,NaN,NaN,NaN,NaN
107,BRK,NJN,1979,1.0,0.0,NaN,NaN,0.000000
108,BRK,NJN,1980,0.0,1.0,0.0,NaN,0.588235
109,BRK,NJN,1981,0.0,0.0,1.0,0.0,0.333333
110,BRK,NJN,1982,1.0,0.0,0.0,1.0,0.190476
111,BRK,NJN,1983,1.0,1.0,0.0,0.0,0.476190
112,BRK,NJN,1984,1.0,1.0,1.0,0.0,0.809524
113,BRK,NJN,1985,1.0,1.0,1.0,1.0,1.000000
114,BRK,NJN,1986,1.0,1.0,1.0,1.0,1.000000
115,BRK,NJN,1987,0.0,1.0,1.0,1.0,1.000000


In [44]:
nba_ml_dataset.to_csv(
    DATA_DIR / "NBA_ML_Dataset.csv",
    index=False
)